In [1]:
!pip install torch torchvision torchaudio
!pip install torch-geometric

SyntaxError: invalid syntax (<ipython-input-1-007be02f099e>, line 1)

In [2]:
# PyTorch Geometric dependencies
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.1.0+cpu.html
!pip install torch-geometric


Looking in links: https://data.pyg.org/whl/torch-2.1.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.4/500.4 kB 773.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.1/753.1 kB 37.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.3/210.3 kB 13.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.8 MB/s eta 0:00:00


In [3]:
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

# تحميل بيانات Cora مع تطبيع الخصائص
dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())
data = dataset[0]  # لأن Cora تحتوي على رسم بياني واحد فقط


/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/libpyg.so: undefined symbol: _ZN2at4_ops10zeros_like4callERKNS_6TensorEN3c108optionalINS5_10ScalarTypeEEENS6_INS5_6LayoutEEENS6_INS5_6DeviceEEENS6_IbEENS6_INS5_12MemoryFormatEEE
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/torch_scatter/_version_cpu.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/usr/local/lib/python3.11/dist-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-

In [4]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv
import torch.nn as nn

class GATNet(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super(GATNet, self).__init__()
        # أول طبقة GAT بعدد رؤوس متعدد
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        # ثاني طبقة GAT (رأس واحد فقط)
        self.gat2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)
        # يمكنك إضافة طبقة خطية إضافية لو أردت
        # self.linear = nn.Linear(out_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.gat2(x, edge_index)
        return F.log_softmax(x, dim=1)


In [5]:
def train(model, data, optimizer, criterion, epochs=200):
    model.train()
    for epoch in range(1, epochs+1):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            val_acc = evaluate(model, data, mask=data.val_mask)
            print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}")


In [6]:
def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out[mask].argmax(dim=1)
        correct = pred.eq(data.y[mask]).sum().item()
        acc = correct / mask.sum().item()
    return acc

def test(model, data):
    test_acc = evaluate(model, data, mask=data.test_mask)
    print(f"Test Accuracy: {test_acc:.4f}")


In [7]:
model = GATNet(dataset.num_node_features, hidden_channels=8, out_channels=dataset.num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

train(model, data, optimizer, criterion)
test(model, data)


Epoch 010, Loss: 1.8570, Val Acc: 0.5340
Epoch 020, Loss: 1.7107, Val Acc: 0.7900
Epoch 030, Loss: 1.4837, Val Acc: 0.7660
Epoch 040, Loss: 1.1990, Val Acc: 0.7900
Epoch 050, Loss: 0.9068, Val Acc: 0.7960
Epoch 060, Loss: 0.6634, Val Acc: 0.7880
Epoch 070, Loss: 0.4951, Val Acc: 0.7800
Epoch 080, Loss: 0.3901, Val Acc: 0.7760
Epoch 090, Loss: 0.3228, Val Acc: 0.7760
Epoch 100, Loss: 0.2754, Val Acc: 0.7540
Epoch 110, Loss: 0.2367, Val Acc: 0.7580
Epoch 120, Loss: 0.2069, Val Acc: 0.7580
Epoch 130, Loss: 0.1848, Val Acc: 0.7540
Epoch 140, Loss: 0.1680, Val Acc: 0.7320
Epoch 150, Loss: 0.1553, Val Acc: 0.7240
Epoch 160, Loss: 0.1450, Val Acc: 0.7160
Epoch 170, Loss: 0.1362, Val Acc: 0.7160
Epoch 180, Loss: 0.1287, Val Acc: 0.7000
Epoch 190, Loss: 0.1221, Val Acc: 0.6960
Epoch 200, Loss: 0.1160, Val Acc: 0.6920
Test Accuracy: 0.7260


In [8]:
from torch_geometric.nn import GCNConv

class GCNNet(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCNNet, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)
